In [2]:
import tarfile
import os

def create_model_package(model_files, inference_script, output_path):
    with tarfile.open(output_path, "w:gz") as tar:
        for file in model_files:
            print(os.path.basename(file))
            tar.add(file, arcname=os.path.join("code", os.path.basename(file)))
        # tar.add(inference_script)# arcname="code/inference.py")

# Usage
model_files = ["../blockhouse_ml","../inference.py", "requirements.txt"]
inference_script = ""
output_path = "model.tar.gz"
create_model_package(model_files, inference_script, output_path)


blockhouse_ml
inference.py
requirements.txt


In [ ]:
# !tar -xvf model.tar.gz

In [3]:
import boto3

s3_client = boto3.client('s3')

version = '1'
bucket_name = 'sagemaker-tradingmodel-us-east-1'
s3_key = f'production/version_{version}/model.tar.gz'

s3_client.upload_file(output_path, bucket_name, s3_key)

In [ ]:
# ccreate session if local
local = False
import boto3
from sagemaker.local import LocalSession
if local:
    sagemaker_session = LocalSession()
    boto_session = sagemaker_session.boto_session
else:
    boto_session = boto3.Session()

In [ ]:

def create_sagemaker_model(model_name, role_arn, image_uri, model_data_url):
    sagemaker_client = boto_session.client('sagemaker')
    
    response = sagemaker_client.create_model(
        ModelName=model_name,
        PrimaryContainer={
            'Image': image_uri,
            'ModelDataUrl': model_data_url,
            'Environment': {
                'SAGEMAKER_PROGRAM': 'inference.py',
                'SAGEMAKER_CONTAINER_LOG_LEVEL': '10',
                'PYTHONUNBUFFERED':'1',
                # refernce :https://github.com/aws/sagemaker-python-sdk/issues/2574#issuecomment-1854585409
                'SAGEMAKER_TS_RESPONSE_TIMEOUT': '600', # 10 minutes
                'SAGEMAKER_MODEL_SERVER_TIMEOUT': '600', # 10 minutes
                # 'SAGEMAKER_SUBMIT_DIRECTORY': '/opt/ml'
                'S3_BUCKET': bucket_name,
                'PROD_VERSION': version,
            }
            
        },
        ExecutionRoleArn=role_arn,
        # EnableNetworkIsolation=False,
        # Mode="SingleModel",
    )
    
    return response

# Usage
model_name = 'model-real-time-inference3'
role_arn = "arn:aws:iam::433046797920:role/service-role/AmazonSageMaker-ExecutionRole-20240227T184832" #'arn:aws:iam::your-account-id:role/your-sagemaker-role'
image_uri = '433046797920.dkr.ecr.us-east-1.amazonaws.com/blockhouse-ml:deploy-test3'
model_data_url = f's3://{bucket_name}/{s3_key}'

response = create_sagemaker_model(model_name, role_arn, image_uri, model_data_url)
print(f"Model created: {response['ModelArn']}")

In [ ]:
def create_sagemaker_endpoint_config(model_name,  endpoint_config_name, endpoint_name, role_arn):
    sagemaker_client = boto_session.client('sagemaker')
    
    variant_name = 'variant-1'

    response = sagemaker_client.create_endpoint_config(
        EndpointConfigName=endpoint_config_name,
        ExecutionRoleArn = role_arn,
        ProductionVariants=[
            {
                'InstanceType': 'ml.g4dn.xlarge',
                'InitialInstanceCount': 1,
                "ManagedInstanceScaling": { 
                    "MaxInstanceCount": 1,
                    "MinInstanceCount": 1,
                },
                "VariantName": variant_name,
            }
        ]
    )
    
    sagemaker_client.create_endpoint(
        EndpointName=endpoint_name,
        EndpointConfigName=endpoint_config_name)
    
    response = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
    
    sagemaker_client.create_inference_component(
        InferenceComponentName=model_name+"-inference-component",
        EndpointName=endpoint_name,
        VariantName=variant_name,
        Specification={
            "ModelName": model_name, 
            "ComputeResourceRequirements": { 
                "NumberOfAcceleratorDevicesRequired": 1,
                "NumberOfCpuCoresRequired": 1, 
                "MinMemoryRequiredInMb": 128,
            }
        },
        RuntimeConfig={"CopyCount": 1},
    )
    # response["EndpointStatus"]
    return response

# Usage
model_name = 'model-real-time-inference3'
endpoint_config_name = 'endpoint-config-real-time-inference3'
endpoint_name = 'endpoint-real-time-inference3'

response = create_sagemaker_endpoint_config(model_name, endpoint_config_name, endpoint_name, role_arn)
print(f"Endpoint created: {response['EndpointArn']}")

In [ ]:
sagemaker_client = boto_session.client('sagemaker')
response = "Creating"
# while response == "Creating":
response = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
# if response["EndpointStatus"] == 'Creating': print(response["EndpointStatus"])
response = response["EndpointStatus"]
print(response)

In [ ]:
## Run inference from endpoint (AWS) real time inference, this code will be used by front end app
import json

# endpoint_name = "Model-1728065209105-deploy-test3-Endpoint-20241005-000648"

# Prepare your JSON payload
payload = {
        "ticker": "AAPL",
        "action": "sell",
        "inventory": 5000,
        "timeframe": 390
    }
request_body = json.dumps(payload)
# Create a low-level client representing Amazon SageMaker Runtime
sagemaker_runtime = boto3.client(
    "sagemaker-runtime", region_name='us-east-1')
# Make the prediction

# Gets inference from the model hosted at the specified endpoint:
response = sagemaker_runtime.invoke_endpoint(
    EndpointName=endpoint_name, 
    Body=request_body, #bytes(request_body, 'utf-8')
    ContentType='application/json',
    InferenceComponentName=model_name+"-inference-component"
    )

# Decodes and prints the response body:
print(response['Body'].read().decode('utf-8'))

In [1]:
# Inference testing locally....
import inference
import json
import os


version = '1'
bucket_name = 'sagemaker-tradingmodel-us-east-1'
os.environ['S3_BUCKET'] = bucket_name
os.environ['PROD_VERSION'] = version

model_dir = 'production_test'
os.makedirs(model_dir, exist_ok=True)

# Load the models from S3 and local storage
model = inference.model_fn(model_dir)

# Simulate a request body
request_body = json.dumps({
    "ticker": "JPM",
    "action": "sell",
    "inventory": 500,
    "timeframe": 300
})

import time
start_t = time.perf_counter()
# Run the inference process
input_data = inference.input_fn(request_body, 'application/json')
prediction = inference.predict_fn(input_data, model)
output = inference.output_fn(prediction, 'application/json')
end_t = time.perf_counter()
print(f"Time taken: {end_t - start_t}")

print("Prediction Result:", output)

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


Initialized sell models at production_test\SellEquityModels and production_test\MicroSellEquityModels


INFO:inference:Initialized sell models at production_test\SellEquityModels and production_test\MicroSellEquityModels


Initialized buy models at production_test\BuyEquityModels and production_test\MicroBuyEquityModels


INFO:inference:Initialized buy models at production_test\BuyEquityModels and production_test\MicroBuyEquityModels


...

inside ohlcv
0.9281227000174113
Got the response
LOGGING: Adding Forecasts to data...
defaulting  3458211368924.0
Ticker found, fetching market cap...
Market cap for JPM: 600130595866.11
LOGGING: Generating Schedule...
large cap, medium-large scenario model selected
model path for macro  production_test\SellEquityModels/model_large_cap_medium_large.h5
------------------------------------------------Class resetted------------------------------------------------


c:\Users\yashv\miniconda3\envs\MLProj\lib\site-packages\arch\univariate\base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
c:\Users\yashv\miniconda3\envs\MLProj\lib\site-packages\arch\univariate\base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
c:\Users\yashv\miniconda3\envs\MLProj\lib\site-packages\arch\univariate\base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
c:\Users\yashv\miniconda3\envs\MLProj\lib\site-packages\arch\univariate\base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

 

--------------------------------------------------
Timestamp: 2024-10-09 13:30:00+00:00, Action: [ 0.17727126 30.        ], Shares: 45, Inventory: 455, TimeLeft: 270
Timestamp: 2024-10-09 14:03:00+00:00, Action: [ 0.17157845 32.11916   ], Shares: 79, Inventory: 376, TimeLeft: 237
Timestamp: 2024-10-09 14:33:00+00:00, Action: [ 0.2060438 30.       ], Shares: 78, Inventory: 298, TimeLeft: 207
Timestamp: 2024-10-09 15:08:00+00:00, Action: [ 0.17    34.17783], Shares: 51, Inventory: 247, TimeLeft: 172
Timestamp: 2024-10-09 15:38:00+00:00, Action: [ 0.1994322 30.       ], Shares: 60, Inventory: 187, TimeLeft: 142
Timestamp: 2024-10-09 16:14:00+00:00, Action: [ 0.18537253 35.114708  ], Shares: 50, Inventory: 137, TimeLeft: 106
Timestamp: 2024-10-09 16:47:00+00:00, Action: [ 0.17     32.765926], Shares: 42, Inventory: 95, TimeLeft: 73
Timestamp: 2024-10-09 17:17:00+00:00, Action: [ 0.17 30.  ], Shares: 37, Inventory: 58, TimeLeft: 43
Timestamp: 2024-10-09 17:48:00+00:00, Action: [ 0.17781846 

INFO:inference:Inference result: [{'step': 1, 'timestamp': '2024-10-09T09:30:00-04:00', 'order_type': 'Market', 'volume': 45, 'limit_price': 210.82278524129055}, {'step': 2, 'timestamp': '2024-10-09T10:03:00-04:00', 'order_type': 'Limit', 'volume': 79, 'limit_price': 211.95241270238034}, {'step': 3, 'timestamp': '2024-10-09T10:33:00-04:00', 'order_type': 'Market', 'volume': 78, 'limit_price': 210.9684844060992}, {'step': 4, 'timestamp': '2024-10-09T11:08:00-04:00', 'order_type': 'Market', 'volume': 51, 'limit_price': 211.04133398850357}, {'step': 5, 'timestamp': '2024-10-09T11:38:00-04:00', 'order_type': 'Market', 'volume': 60, 'limit_price': 211.1141835709079}, {'step': 6, 'timestamp': '2024-10-09T12:14:00-04:00', 'order_type': 'Limit', 'volume': 50, 'limit_price': 212.24488780522867}, {'step': 7, 'timestamp': '2024-10-09T12:47:00-04:00', 'order_type': 'Market', 'volume': 42, 'limit_price': 211.25988273571656}, {'step': 8, 'timestamp': '2024-10-09T13:17:00-04:00', 'order_type': 'Marke

Time taken: 7.833776999992551
Prediction Result: [{"step": 1, "timestamp": "2024-10-09T09:30:00-04:00", "order_type": "Market", "volume": 45, "limit_price": 210.82278524129055}, {"step": 2, "timestamp": "2024-10-09T10:03:00-04:00", "order_type": "Limit", "volume": 79, "limit_price": 211.95241270238034}, {"step": 3, "timestamp": "2024-10-09T10:33:00-04:00", "order_type": "Market", "volume": 78, "limit_price": 210.9684844060992}, {"step": 4, "timestamp": "2024-10-09T11:08:00-04:00", "order_type": "Market", "volume": 51, "limit_price": 211.04133398850357}, {"step": 5, "timestamp": "2024-10-09T11:38:00-04:00", "order_type": "Market", "volume": 60, "limit_price": 211.1141835709079}, {"step": 6, "timestamp": "2024-10-09T12:14:00-04:00", "order_type": "Limit", "volume": 50, "limit_price": 212.24488780522867}, {"step": 7, "timestamp": "2024-10-09T12:47:00-04:00", "order_type": "Market", "volume": 42, "limit_price": 211.25988273571656}, {"step": 8, "timestamp": "2024-10-09T13:17:00-04:00", "ord

In [ ]:
from sagemaker import image_uris
image_uris.retrieve(framework='pytorch',region='us-east-1',version='2.2.0',py_version='py310',image_scope='inference', instance_type='ml.g4dn.xlarge')



In [25]:
!python deployment_async.py --model_name "model-async-inference-v1" \
        --endpoint_config_name "endpoint-config-async-inference-v1" \
        --endpoint_name "endpoint-async-inference-v1" \
        --sagemaker_role "arn:aws:iam::433046797920:role/service-role/AmazonSageMakerServiceCatalogProductsUseRole" \
        --instance_type "ml.m5.xlarge" \
        --version 1 \
        --model_files "../blockhouse_ml" "./inference.py" "requirements.txt" \
        --account_id "433046797920" \
        --input_s3_location "s3://sagemaker-tradingmodel-us-east-1/csv_dump/corrected_dates_output_just_2.csv" \
        --output_s3_location "s3://sagemaker-tradingmodel-us-east-1/predictions/"


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/base_serializers.py:28: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 1.22.4)
  import scipy.sparse
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
INFO:__main__:Packaging files: ['../blockhouse_ml', './inference.py', 'requirements.txt']
INFO:__main__:Adding file: blockhouse_ml
INFO:__main__:Adding file: inference.py
INFO:__main__:Adding file: requirements.txt
INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
INFO:__main__:Uploading model package to s3://sagemaker-tradingmodel-us-east-1/production/version_1/mode-async.tar.gz
INFO:__main__:Model model-async-inference-v1 created successfully.
INFO:__main__:Model created: arn:aws:sagemaker:us-east-1:433046

In [26]:
import boto3
import json

sagemaker_runtime = boto3.client('sagemaker-runtime')

endpoint_name = 'endpoint-async-inference-v1'
input_location = 's3://sagemaker-tradingmodel-us-east-1/csv_dump/corrected_dates_output_just_2.csv'

# Invoke the endpoint asynchronously
response = sagemaker_runtime.invoke_endpoint_async(
    EndpointName=endpoint_name,
    InputLocation=input_location,
    ContentType='text/csv'  # Change this to match your input data format
)

# Get the output location
output_location = response['OutputLocation']
print(f"Asynchronous inference job submitted. Output will be available at: {output_location}")

Asynchronous inference job submitted. Output will be available at: s3://sagemaker-tradingmodel-us-east-1/predictions/3c747f9c-f6d2-4d03-9ac5-6893c896c297.out


In [27]:
async_inference_job_name = response['InferenceId'] 

In [10]:
while True:
    job_done = check_async_inference_status(endpoint_name, async_inference_job_name)
    if job_done:
        break
    time.sleep(60)  # Wait for 60 seconds before checking again

logger.info("Async inference job completed.")

NameError: name 'check_async_inference_status' is not defined

In [15]:
import boto3
import time

sagemaker_client = boto3.client('sagemaker')

while True:
    response = sagemaker_client.describe_inference_job(InferenceJobName=job_id)
    status = response['InferenceJobStatus']
    if status == 'Completed':
        print("Job completed. Retrieving results...")
        break
    elif status in ['Failed', 'Stopped']:
        print(f"Job {status}. Check SageMaker console for details.")
        break
    print("Job still in progress. Waiting...")
    time.sleep(30)  # Wait for 30 seconds before checking again

AttributeError: 'SageMaker' object has no attribute 'describe_inference_job'

In [ ]:
import boto3
import pandas as pd

s3 = boto3.client('s3')

# Parse the S3 URI
bucket_name, key = output_location.replace("s3://", "").split("/", 1)

# Download the results file
local_file = '/tmp/results.csv'
s3.download_file(bucket_name, key, local_file)

# Read and process the results
results_df = pd.read_csv(local_file)
print(results_df.head())

In [28]:
import boto3

# Create a SageMaker runtime client
sagemaker_runtime = boto3.client('sagemaker-runtime')

# Define your endpoint name and S3 paths
endpoint_name = 'endpoint-async-inference-v1'
input_s3_path = 's3://sagemaker-tradingmodel-us-east-1/csv_dump/your-input-file.csv'  # Update with your actual input file path
output_s3_path = 's3://sagemaker-tradingmodel-us-east-1/predictions/'

# Invoke the asynchronous endpoint
response = sagemaker_runtime.invoke_endpoint_async(
    EndpointName=endpoint_name,
    InputLocation=input_s3_path,
    OutputLocation=output_s3_path
)

# Get the asynchronous job ID (InferenceId) from the response
async_job_id = response['InferenceId']
print(f"Asynchronous job started with ID: {async_job_id}")


ParamValidationError: Parameter validation failed:
Unknown parameter in input: "OutputLocation", must be one of: EndpointName, ContentType, Accept, CustomAttributes, InferenceId, InputLocation, RequestTTLSeconds, InvocationTimeoutSeconds

In [31]:
import boto3
import argparse
import logging
import time

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialize the SageMaker runtime client
sagemaker_client = boto3.client('sagemaker-runtime', region_name='us-east-1')

# Set your SageMaker endpoint name and S3 paths
endpoint_name = "endpoint-async-inference-v1"  # Your endpoint name
input_s3_bucket = "sagemaker-tradingmodel-us-east-1"  # Your input S3 bucket
input_s3_key = "csv_dump/corrected_dates_output.csv"  # The input file path in S3

def start_async_inference_job():
    """
    Function to start an async inference job in SageMaker using invoke_endpoint_async.
    """
    response = sagemaker_client.invoke_endpoint_async(
        EndpointName=endpoint_name,
        InputLocation=f"s3://{input_s3_bucket}/{input_s3_key}",
        ContentType='text/csv'  # Ensure content type matches your input data
    )
    
    logger.info(f"Started async inference job: {response['InferenceId']}")
    return response['InferenceId']

if __name__ == "__main__":
    inference_id = start_async_inference_job()
    logger.info(f"Async Inference Job ID: {inference_id}")

INFO:__main__:Started async inference job: 879d4407-b471-44c5-a261-7b72db9acb67
INFO:__main__:Async Inference Job ID: 879d4407-b471-44c5-a261-7b72db9acb67


In [30]:
start_async_inference_job

<function __main__.start_async_inference_job()>

In [3]:
import boto3
import time
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialize the SageMaker and S3 clients
sagemaker_runtime = boto3.client('sagemaker-runtime')
sagemaker_client = boto3.client('sagemaker')
s3_client = boto3.client('s3')

def check_async_inference_status(endpoint_name, inference_id):
    # First, get the endpoint configuration to find the output S3 location
    response = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
    endpoint_config_name = response['EndpointConfigName']
    
    config_response = sagemaker_client.describe_endpoint_config(EndpointConfigName=endpoint_config_name)
    output_config = config_response['AsyncInferenceConfig']['OutputConfig']
    s3_output_path = output_config['S3OutputPath']
    
    # Construct the expected S3 key for the output
    bucket, key_prefix = s3_output_path.replace("s3://", "").split("/", 1)
    output_key = f"{key_prefix}/{inference_id}/output"
    
    try:
        # Check if the output file exists
        s3_client.head_object(Bucket=bucket, Key=output_key)
        logger.info(f"Inference job {inference_id} completed. Results available in S3.")
        return True
    except s3_client.exceptions.ClientError as e:
        if e.response['Error']['Code'] == '404':
            logger.info(f"Inference job {inference_id} still in progress.")
            return False
        else:
            # If there's an error other than 404, raise it
            raise

def start_async_inference_job(endpoint_name, input_location):
    response = sagemaker_runtime.invoke_endpoint_async(
        EndpointName=endpoint_name,
        InputLocation=input_location,
        ContentType='text/csv'
    )
    return response['InferenceId']

if __name__ == "__main__":
    endpoint_name = "endpoint-async-inference-v1"  # Replace with your endpoint name
    input_s3_path = 's3://sagemaker-tradingmodel-us-east-1/csv_dump/corrected_dates_output_just_2.csv'  # Replace with your input S3 path
    
    # Start the async inference job
    inference_id = start_async_inference_job(endpoint_name, input_s3_path)
    logger.info(f"Started async inference job: {inference_id}")
    
    # Poll the job status until it completes or fails
    while True:
        job_done = check_async_inference_status(endpoint_name, inference_id)
        if job_done:
            break
        time.sleep(60)  # Wait for 60 seconds before checking again

    logger.info("Async inference job completed.")

INFO:__main__:Started async inference job: a46f993b-2716-4e2a-b430-0958711d624d
INFO:__main__:Inference job a46f993b-2716-4e2a-b430-0958711d624d still in progress.
INFO:__main__:Inference job a46f993b-2716-4e2a-b430-0958711d624d still in progress.
INFO:__main__:Inference job a46f993b-2716-4e2a-b430-0958711d624d still in progress.
INFO:__main__:Inference job a46f993b-2716-4e2a-b430-0958711d624d still in progress.
INFO:__main__:Inference job a46f993b-2716-4e2a-b430-0958711d624d still in progress.
INFO:__main__:Inference job a46f993b-2716-4e2a-b430-0958711d624d still in progress.
INFO:__main__:Inference job a46f993b-2716-4e2a-b430-0958711d624d still in progress.
INFO:__main__:Inference job a46f993b-2716-4e2a-b430-0958711d624d still in progress.
INFO:__main__:Inference job a46f993b-2716-4e2a-b430-0958711d624d still in progress.


KeyboardInterrupt: 

In [2]:
!python deployment_async.py --model_name "model-async-inference-adjustedVWAP" \
        --endpoint_config_name "endpoint-config-async-inference-adjustedVWAP" \
        --endpoint_name "endpoint-async-inference-adjustedVWAP" \
        --sagemaker_role "arn:aws:iam::433046797920:role/service-role/AmazonSageMakerServiceCatalogProductsUseRole" \
        --instance_type "ml.g4dn.xlarge" \
        --version 1 \
        --model_files "../Adjusted_VWAP" "./inference.py" "../Adjusted_VWAP/requirements.txt" \
        --account_id "433046797920" \
        --input_s3_location "s3://sagemaker-tradingmodel-us-east-1/csv_dump/corrected_dates_output_just_2.csv" \
        --output_s3_location "s3://sagemaker-tradingmodel-us-east-1/predictions/"


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
INFO:__main__:Packaging files: ['../Adjusted_VWAP', './inference.py', '../Adjusted_VWAP/requirements.txt']
INFO:__main__:Adding file: Adjusted_VWAP
INFO:__main__:Adding file: inference.py
INFO:__main__:Adding file: requirements.txt
INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
INFO:__main__:Uploading model package to s3://sagemaker-tradingmodel-us-east-1/production/version_1/model-async-adjustedVWAP.tar.gz
INFO:__main__:Model model-async-inference-adjustedVWAP created successfully.
INFO:__main__:Model created: arn:aws:sagemaker:us-east-1:433046797920:model/model-async-inference-adjustedVWAP
INFO:__main__:Endpoint endpoint-async-inference-adjustedVWAP created. Waiting for it to be 'InService'...
INFO:__main__:Current endpoint status: Cr

In [4]:
import boto3
import json

sagemaker_runtime = boto3.client('sagemaker-runtime')

endpoint_name = 'endpoint-async-inference-adjustedVWAP'
input_location = 's3://sagemaker-tradingmodel-us-east-1/csv_dump/corrected_dates_output.csv'

# Invoke the endpoint asynchronously
response = sagemaker_runtime.invoke_endpoint_async(
    EndpointName=endpoint_name,
    InputLocation=input_location,
    ContentType='text/csv'  # Change this to match your input data format
)

# Get the output location
output_location = response['OutputLocation']
print(f"Asynchronous inference job submitted. Output will be available at: {output_location}")

Asynchronous inference job submitted. Output will be available at: s3://sagemaker-tradingmodel-us-east-1/predictions/e6890ac3-20cd-41ad-967f-d294ed8c23de.out
